In [1]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDFS_PATH = str(REPO_ROOT / "data" / "mcq" / "pdf")
OUTPUT_FILE = "./questions.json"
MIN_IMAGE_BYTES = 5000

In [2]:
# Qwen3.5-27B block parser: reads one question block's raw lines and emits structured
# stem/options/answer/reference in a single pass. Same model/loading pattern as the
# triple extractor in notebooks/mcq_generation.ipynb (src/captioning/qwen_vl.py lineage) —
# lazy load/unload, one model resident at a time on the unified-memory budget.
#
# Qwen3.5 is a multimodal checkpoint (AutoModelForMultimodalLM + AutoProcessor rather than
# AutoModelForCausalLM + AutoTokenizer) but runs fine text-only — just omit image content
# from the messages.
#
# enable_thinking=False: this is structured extraction, not open-ended reasoning.

import gc

import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_PATH = "Qwen/Qwen3.5-9B"


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class BlockParser:
    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.model_path,
            dtype=torch.bfloat16,
        )
        self.model.to(self.device)
        self.model.eval()

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def generate(self, system_prompt: str, user_content: str, max_new_tokens: int = 1200) -> str:
        assert self.model is not None, "call load() first"
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


block_parser = BlockParser()
block_parser.load()
print(f"Loaded {MODEL_PATH} on {block_parser.device}")

/opt/jupyter-homes/u257878269/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0803 10:50:58.946000 67250 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 760/760 [00:00<00:00, 8500.75it/s]


Loaded Qwen/Qwen3.5-9B on mps


In [3]:
import re

OPTION_RE = re.compile(r'^([A-Ea-e])[.)\s]\s*(.+)$', re.DOTALL)
ANSWER_RE = re.compile(r'^ANSWER\s*:\s*([A-Ea-e])', re.IGNORECASE)
NUM_SOLO = re.compile(r'^\d{1,2}$')
NUM_LEAD = re.compile(r'^(\d{1,2})[.)\s]\s*(.+)$', re.DOTALL)
REF_RE = re.compile(r'^Referensi\s*[:\-]?\s*', re.IGNORECASE)

In [4]:
import json
import fitz
from tqdm.auto import tqdm

# ── image extraction ──────────────────────────────────────────────────────
def save_images(doc, out_dir: Path, stem: str) -> list:
    saved, seen = [], set()
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    for pnum, page in enumerate(doc):
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in seen: continue
            seen.add(xref)
            try:
                bi = doc.extract_image(xref)
                if not bi or bi["size"] < MIN_IMAGE_BYTES: continue
                rects = page.get_image_rects(xref)
                fname = f"{stem}_p{pnum+1}_x{xref}.{bi['ext']}"
                fpath = out_dir / "images" / fname
                fpath.write_bytes(bi["image"])
                saved.append({
                    "page": pnum,
                    "y": rects[0].y0 if rects else 0,
                    "path": f"images/{fname}",
                })
            except Exception:
                pass
    return saved


# ── line extraction ───────────────────────────────────────────────────────
def get_lines(doc, images: list) -> list:
    items = []
    for pnum, page in enumerate(doc):
        for blk in page.get_text("dict")["blocks"]:
            if blk["type"] != 0: continue
            for ln in blk["lines"]:
                spans = ln["spans"]
                txt = "".join(s["text"] for s in spans).strip()
                if not txt: continue
                items.append({
                    "t": txt,
                    "bold": any(bool(s["flags"] & 16) for s in spans),
                    "page": pnum,
                    "y": spans[0]["bbox"][1],
                    "img": False,
                    "img_idx": None,
                })
    for i, img in enumerate(images):
        items.append({"t": f"__IMG_{i}__", "bold": False,
                      "page": img["page"], "y": img["y"],
                      "img": True, "img_idx": i})
    items.sort(key=lambda x: (x["page"], x["y"]))
    return items


# ── format detection ──────────────────────────────────────────────────────
def detect_fmt(full_text: str, lines: list) -> str:
    if re.search(r"ANSWER\s*:", full_text, re.IGNORECASE):
        return "A"
    if any(l["bold"] and OPTION_RE.match(l["t"]) for l in lines):
        return "B"
    return "C"


# ── segment into question blocks ──────────────────────────────────────────
def segment(lines: list) -> list:
    blocks, cur = [], []
    for item in lines:
        t = item["t"].strip()
        if NUM_SOLO.match(t):                          # standalone "1", "2"…
            if cur: blocks.append(cur)
            cur = []; continue
        m = NUM_LEAD.match(t)
        if m and int(m.group(1)) <= 60:
            remainder = m.group(2).strip()
            if not OPTION_RE.match(remainder):         # "1. Seorang…" style
                if cur: blocks.append(cur)
                cur = []
                item = dict(item, t=remainder)
        cur.append(item)
    if cur: blocks.append(cur)
    # keep only blocks that have at least one A-E option
    return [b for b in blocks
            if any(OPTION_RE.match(i["t"]) for i in b if not i["img"])]


# ── LLM-based block parsing ───────────────────────────────────────────────
# Regex segmentation (numbered blocks, A-E option lines) is reliable — only the
# stem/reference/answer split inside a block is fragile (bold cues, ANSWER:
# lines, "Referensi" wording all vary across PDFs). An LLM call per block
# replaces that split entirely instead of growing more regex special-cases.

PARSE_SYSTEM_PROMPT = """You extract structured data from one Indonesian medical multiple-choice question block that was OCR/text-extracted from a PDF. The raw lines may have the stem, a reference/citation section, and options interleaved or out of order due to PDF text extraction quirks.

Return ONLY a JSON object, no markdown fences, no commentary, with this exact shape:
{
  "stem": "the question text only, no header noise like 'Paket 3' or 'Soal:', no reference/citation text",
  "options": {"A": "...", "B": "...", ...},
  "answer": "A" | null,
  "reference": "citation/source text if present, else empty string"
}

Rules:
- "answer" is the correct option letter if explicitly marked (e.g. "ANSWER: C", or bold option text prefixed with [BOLD] below). Use null if no marker is present — do not guess.
- Options must be copied verbatim, just trimmed (strip any leading "[BOLD] " marker from the text).
- If reference text and stem text got merged together, split them: reference is usually a book/journal citation (e.g. "Referensi: ..." author/year/page), stem is the clinical vignette/question (often starts with "Seorang", "Seseorang", "Pasien", "Apakah", "Bagaimana", "Manakah", "Pada ...").
- Ignore lines that are just page numbers, "Paket X", or image placeholders like __IMG_0__."""

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def _repair_truncated_json(raw: str) -> str:
    """Best-effort fix for JSON cut off mid-generation (max_new_tokens truncation): drops
    back to the last fully-formed value (closing an unterminated string first if needed),
    then closes whatever braces/brackets are still open — in the correct nesting order —
    so json.loads has a chance at the still-complete fields."""
    text = raw[raw.index("{"):] if "{" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    last_safe = max(text.rfind("}"), text.rfind("]"), text.rfind(","))
    if last_safe == -1:
        raise ValueError("no safe truncation point found")
    text = text[:last_safe] if text[last_safe] == "," else text[: last_safe + 1]

    stack: list[str] = []
    in_string = False
    escape = False
    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            stack.pop()

    closers = {"{": "}", "[": "]"}
    return text + "".join(closers[ch] for ch in reversed(stack))


def _call_llm_json(user_content: str, max_tokens: int = 1200) -> dict:
    raw = block_parser.generate(PARSE_SYSTEM_PROMPT, user_content, max_new_tokens=max_tokens)

    # Qwen3.5 is a reasoning model — strip a <think>...</think> block if the
    # thinking-disable chat template kwarg didn't take, so stray braces inside the
    # reasoning trace don't get matched instead of the real JSON answer.
    cleaned = _THINK_BLOCK_RE.sub("", raw)
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned.strip())

    matches = _JSON_BLOCK_RE.findall(cleaned)
    if not matches:
        raise ValueError(f"No JSON object found in LLM output: {raw[:200]!r}")
    candidate = matches[-1]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return json.loads(_repair_truncated_json(candidate))


# ── parse one block → question dict ──────────────────────────────────────
def parse_block(block, fmt, images, stem, idx):
    img_paths = []
    raw_lines = []
    for item in block:
        t = item["t"].strip()
        if not t: continue
        if item["img"]:
            ii = item["img_idx"]
            if ii is not None and ii < len(images):
                img_paths.append(images[ii]["path"])
            raw_lines.append(t)  # keep __IMG_i__ marker inline for context
            continue
        prefix = "[BOLD] " if (item["bold"] and fmt == "B") else ""
        raw_lines.append(prefix + t)

    block_text = "\n".join(raw_lines)

    try:
        parsed = _call_llm_json(block_text)
    except Exception as e:
        tqdm.write(f"  [warn] LLM parse failed for {stem}_Q{idx:03d}: {e}")
        parsed = {"stem": "", "options": {}, "answer": None, "reference": ""}

    options = {str(k).upper(): str(v).strip()
               for k, v in (parsed.get("options") or {}).items()
               if str(k).upper() in "ABCDE"}
    answer = parsed.get("answer")
    if answer:
        answer = str(answer).strip().upper()
        if answer not in "ABCDE" or answer not in options:
            answer = None

    return {
        "id"         : f"{stem}_Q{idx:03d}",
        "source"     : f"{stem}.pdf",
        "format"     : fmt,
        "stem"       : str(parsed.get("stem") or "").strip(),
        "options"    : {k: options[k] for k in sorted(options)},
        "answer"     : answer,           # null when not detected
        "reference"  : str(parsed.get("reference") or "").strip(),
        "has_image"  : bool(img_paths),
        "image_paths": img_paths,
    }


# ── main function ─────────────────────────────────────────────────────────
def extract_pdf(pdf_path: str, out_dir: Path) -> list:
    file_stem = Path(pdf_path).stem
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        tqdm.write(f"  [skip] {Path(pdf_path).name}: {e}")
        return []

    images    = save_images(doc, out_dir, file_stem)
    lines     = get_lines(doc, images)
    full_text = "\n".join(p.get_text() for p in doc)
    fmt       = detect_fmt(full_text, lines)
    blocks    = segment(lines)

    questions = []
    for i, block in enumerate(tqdm(blocks, desc=file_stem, unit="q", leave=False), 1):
        q = parse_block(block, fmt, images, file_stem, i)
        if q["options"] and (q["stem"] or q["has_image"]):
            questions.append(q)

    doc.close()
    return questions

print("Extractor defined.")

Extractor defined.


In [5]:
out_dir = Path(OUTPUT_FILE).parent
pdf_files = sorted(Path(PDFS_PATH).glob("*.pdf"))

if not pdf_files:
    print(f"No PDFs found in {INPUT_DIR}")
else:
    print(f"Found {len(pdf_files)} PDF files\n")

all_questions = []

for pdf in tqdm(pdf_files, desc="PDFs", unit="file"):
    qs    = extract_pdf(str(pdf), out_dir)
    auto  = sum(1 for q in qs if q["answer"])
    imgs  = sum(1 for q in qs if q["has_image"])
    fmt   = qs[0]["format"] if qs else "?"
    tqdm.write(f"  {pdf.name:<45}  Fmt:{fmt}  {len(qs):>2}Q  "
               f"{auto:>2} w/answer  {imgs} img")
    all_questions.extend(qs)

# save
Path(OUTPUT_FILE).write_text(
    json.dumps(all_questions, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

total  = len(all_questions)
w_ans  = sum(1 for q in all_questions if q["answer"])
no_ans = total - w_ans
imgs   = sum(1 for q in all_questions if q["has_image"])

print(f"""
{'='*50}
Total questions  : {total}
With answer      : {w_ans}
Without answer   : {no_ans}  ← fill manually or use LLM
With images      : {imgs}
Output           : {OUTPUT_FILE}
{'='*50}
""")

Found 220 PDF files



PDFs:   0%|          | 1/220 [00:07<28:46,  7.88s/file]        

  Copy of PAKET A31.pdf                          Fmt:?   0Q   0 w/answer  0 img



PDFs:   1%|          | 2/220 [00:16<29:18,  8.07s/file]          

  Copy of Paket A1(1).pdf                        Fmt:A   1Q   1 w/answer  0 img


...

  Salinan Paket C7.pdf                           Fmt:A   5Q   5 w/answer  0 img



PDFs: 100%|█████████▉| 219/220 [1:10:47<00:36, 36.18s/file]   

  Salinan Paket C8.pdf                           Fmt:A   4Q   4 w/answer  0 img



PDFs: 100%|██████████| 220/220 [1:11:30<00:00, 19.50s/file]   

  Salinan Paket C9.pdf                           Fmt:A   5Q   5 w/answer  0 img

Total questions  : 431
With answer      : 405
Without answer   : 26  ← fill manually or use LLM
With images      : 139
Output           : ./questions.json



In [6]:
for q in all_questions[:3]:
    print(f"[{q['id']}]  fmt:{q['format']}  answer:{q['answer']}")
    print(f"  {q['stem'][:100]}...")
    for k, v in q["options"].items():
        marker = " ◀" if k == q["answer"] else ""
        print(f"    {k}. {v[:60]}{marker}")
    print()


[Copy of Paket A1(1)_Q001]  fmt:A  answer:C
  Bagian dari medulla spinalis yang memiliki perikaryon kecil dan berfungsi menerima serabut afferent ...
    A. Cornu Lateralis Substransia Grisea
    B. Cornu Anterior Substransia Grisea
    C. Cornu Posterior Substransia Grisea ◀
    D. Semua bagian substansia Alba
    E. Semua bagian substansia grisea

[Copy of Paket A1_Q001]  fmt:A  answer:C
  Bagian dari medulla spinalis yang memiliki perikaryon kecil dan berfungsi menerima serabut afferent ...
    A. Cornu Lateralis Substransia Grisea
    B. Cornu Anterior Substransia Grisea
    C. Cornu Posterior Substransia Grisea ◀
    D. Semua bagian substansia Alba
    E. Semua bagian substansia grisea

[Copy of Paket A10_Q001]  fmt:A  answer:B
  Apakah peran rangsangan saraf parasimpatis terkait kontrol vesika urinaria?...
    A. Menginhibisi kontraksi otot detrusor selama proses pengisian
    B. Menstimulasi kontraksi otot detrusor selama proses miksturis ◀
    C. Menginhibisi spincter uretra ek